# EEG2GAIT — Hierarchical GCN for EEG-Based Gait Decoding  *(v2)*

Full training pipeline on the MoBI dataset.  
GPU accelerated · HTSR loss · early stopping on val Pearson *r*

> **v2** — regenerated from authoritative `src/` files (June 2026).  
> Architecture: K=2 HGP branches (depths [1,2]), depth-wise GSL, FFN [50,100,200].


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'scikit-learn'],
    check=True
)
print('Dependencies ready.')

## Verify Data Path

In [ ]:
import os
from pathlib import Path

DATA_DIR = Path('/kaggle/input/datasets/jwangldan09/eeg2gait-fall-prediction-dataset/RepositoryData')
OUTPUT_DIR = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    print(f'ERROR: {DATA_DIR} does not exist.')
    print('Contents of /kaggle/input:')
    os.system('find /kaggle/input -maxdepth 3 -type d')
else:
    sessions = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
    print(f'Found {len(sessions)} session folders')
    for s in sessions[:6]:
        print(f'  {s.name}  eeg={(s/"eeg.txt").exists()}  joints={(s/"joints.txt").exists()}')
    if len(sessions) > 6:
        print(f'  ... and {len(sessions)-6} more')

## CONFIG — `config.py`

In [ ]:
%%writefile /kaggle/working/config.py
"""
config.py
---------
Central configuration for the EEG2GAIT pipeline.
All hyper-parameters are defined here so every other module
can import from a single source of truth.
"""

import os
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────────
ROOT_DIR   = Path("/kaggle")
DATA_DIR   = Path("/kaggle/input/datasets/jwangldan09/eeg2gait-fall-prediction-dataset/RepositoryData")
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset ──────────────────────────────────────────────────────────────────────
SUBJECTS = [f"SL{i:02d}" for i in range(1, 9)]   # SL01 … SL08
SESSIONS = ["T01", "T02", "T03"]

# Raw sampling rate (from data inspection: ~333.3 Hz → 1/0.003)
RAW_FS   = 333.33   # Hz (approximate; actual dt = 0.003 s)

# Target sampling rate after downsampling
TARGET_FS = 100     # Hz

# Band-pass filter  [Hz]
BANDPASS_LO = 0.1
BANDPASS_HI = 48.0

# Number of EEG channels
N_CHANNELS_RAW  = 64
EOG_CHAN_INDICES = [32, 38, 39, 62, 63]   # 0-based in the data matrix (after time col)
N_CHANNELS       = 59                     # channels kept

RADIUS_MM = 30.0    # adjacency radius for GCM

# ── Joints ───────────────────────────────────────────────────────────────────────
JOINT_NAMES = ["GHR", "GKR", "GAR", "GHL", "GKL", "GAL"]
N_JOINTS    = 6     # d_j in the paper

# ── MoBI Data-Split (per session, in minutes) ─────────────────────────────
TRAIN_MIN   = 13.5
VAL_MIN     = 1.5
TEST_MIN    = 5.0

# ── Window / stride ──────────────────────────────────────────────────────────────
WINDOW_SECS  = 1.0          # 1-second window
STRIDE_SECS  = 0.1          # 100 ms stride (10-fold overlap)
WINDOW_SAMPS = int(WINDOW_SECS  * TARGET_FS)   # 100 samples
STRIDE_SAMPS = int(STRIDE_SECS * TARGET_FS)    # 10  samples

# ── Model (LTL → GCM → HGP → GSL → FFN → GTL → Output) ──
F_FILTERS       = 25
LTL_KERNEL      = 10
HGP_DEPTHS      = [1, 2]
DROPOUT_P       = 0.5
POOL_WIDTH      = 3
KERNEL_WIDTH    = 10
FFN_FILTERS     = [50, 100, 200]
GTL_HEADS       = 4
GTL_DROPOUT     = 0.1

# ── Training ─────────────────────────────────────────────────────────────────────
BATCH_SIZE     = 100
LR             = 1e-3
MAX_EPOCHS     = 50
PATIENCE       = 30

# ── Loss ─────────────────────────────────────────────────────────────────────────
ALPHA   = 0.5
BETA    = 0.1
EPSILON = 1e-8

# ── Misc ─────────────────────────────────────────────────────────────────────────
SEED        = 42
NUM_WORKERS = 0
DEVICE      = "auto"


## DATASET — `dataset.py`

In [ ]:
%%writefile /kaggle/working/dataset.py
"""
dataset.py
----------
Custom PyTorch Dataset & DataLoader for the MoBI EEG2GAIT dataset.

Preprocessing pipeline (per session):
    1. Read raw EEG + joint angles at ~333 Hz
    2. Drop 5 artifact / EOG channels  -> 59 channels
    3. Band-pass filter  0.1-48 Hz  (scipy butterworth, minimum-phase)
    4. Common-Average Reference (CAR)
    5. Resample to 100 Hz  (scipy.signal.resample_poly)
    6. Align EEG <-> joints by timestamp
    7. Split by time: first 13.5 min -> train, next 1.5 min -> val, last 5 min -> test
    8. Sliding-window extraction: 1-second windows, 100 ms stride
"""

import os
import re
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, sosfilt, resample_poly
from scipy.spatial.distance import cdist
from math import gcd

try:
    from .config import (
        DATA_DIR, SUBJECTS, SESSIONS,
        RAW_FS, TARGET_FS,
        BANDPASS_LO, BANDPASS_HI,
        N_CHANNELS_RAW, EOG_CHAN_INDICES, N_CHANNELS,
        N_JOINTS,
        TRAIN_MIN, VAL_MIN, TEST_MIN,
        WINDOW_SAMPS, STRIDE_SAMPS,
        RADIUS_MM,
    )
except ImportError:
    from config import (
        DATA_DIR, SUBJECTS, SESSIONS,
        RAW_FS, TARGET_FS,
        BANDPASS_LO, BANDPASS_HI,
        N_CHANNELS_RAW, EOG_CHAN_INDICES, N_CHANNELS,
        N_JOINTS,
        TRAIN_MIN, VAL_MIN, TEST_MIN,
        WINDOW_SAMPS, STRIDE_SAMPS,
        RADIUS_MM,
    )


# Helper: I/O

def _read_eeg(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """Return (timestamps [N], data [N, 64]) from eeg.txt."""
    with open(path, "r", errors="replace") as f:
        _ = f.readline()  # skip header
        rows = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = line.split("\t")
            try:
                rows.append([float(v) for v in vals if v])
            except ValueError:
                continue
    arr = np.array(rows, dtype=np.float32)
    timestamps = arr[:, 0]
    data       = arr[:, 1:]
    return timestamps, data


def _read_joints(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return (timestamps [N], joint_angles [N, 6]) from joints.txt.
    Only the first 6 joints (GHR,GKR,GAR,GHL,GKL,GAL) are used.
    """
    with open(path, "r", errors="replace") as f:
        h1 = f.readline()
        h2 = f.readline()
        rows = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = line.split("\t")
            try:
                rows.append([float(v) for v in vals if v])
            except ValueError:
                continue
    arr = np.array(rows, dtype=np.float32)
    timestamps   = arr[:, 0]
    joint_angles = arr[:, 1:7]
    return timestamps, joint_angles


# Helper: Preprocessing

def _drop_eog_channels(data: np.ndarray) -> np.ndarray:
    """Drop EOG/artifact channels; keep 59 of the 64 channels."""
    keep = [i for i in range(N_CHANNELS_RAW) if i not in EOG_CHAN_INDICES]
    return data[:, keep]


def _common_average_reference(data: np.ndarray) -> np.ndarray:
    """Subtract the mean across channels at each time point."""
    return data - data.mean(axis=1, keepdims=True)


def _bandpass_filter(data: np.ndarray, fs: float) -> np.ndarray:
    """Minimum-phase Butterworth band-pass filter (0.1-48 Hz), paper Sec. IV-A."""
    nyq  = fs / 2.0
    lo   = BANDPASS_LO / nyq
    hi   = min(BANDPASS_HI / nyq, 0.999)
    sos  = butter(4, [lo, hi], btype="band", output="sos")
    return sosfilt(sos, data, axis=0).astype(data.dtype)


def _resample(data: np.ndarray, fs_in: float, fs_out: float) -> np.ndarray:
    """Resample from fs_in -> fs_out using polyphase method."""
    fs_in_int  = int(round(fs_in  * 3))
    fs_out_int = int(round(fs_out * 3))
    g          = gcd(fs_in_int, fs_out_int)
    up, down   = fs_out_int // g, fs_in_int // g

    resampled = np.empty((round(data.shape[0] * up / down), data.shape[1]), dtype=np.float32)
    for ch in range(data.shape[1]):
        resampled[:, ch] = resample_poly(data[:, ch], up, down).astype(np.float32)
    return resampled


def _preprocess_session(eeg_raw: np.ndarray, joints_raw: np.ndarray,
                         fs: float) -> Tuple[np.ndarray, np.ndarray]:
    """Full preprocessing pipeline applied to one session."""
    eeg = _drop_eog_channels(eeg_raw)
    eeg = _bandpass_filter(eeg, fs)       # bandpass first (paper Sec. IV-A)
    eeg = _common_average_reference(eeg)  # then CAR
    eeg = _resample(eeg, fs, TARGET_FS)
    joints = _resample(joints_raw, fs, TARGET_FS)
    n = min(len(eeg), len(joints))
    return eeg[:n], joints[:n]


# Helper: Windowing

def _extract_windows(eeg: np.ndarray, joints: np.ndarray,
                     window: int = WINDOW_SAMPS,
                     stride: int = STRIDE_SAMPS
                     ) -> Tuple[np.ndarray, np.ndarray]:
    """Sliding-window extraction."""
    T = eeg.shape[0]
    indices = list(range(0, T - window + 1, stride))
    X_list, y_list = [], []
    for s in indices:
        e = s + window
        X_list.append(eeg[s:e].T)
        y_list.append(joints[s:e].mean(0))
    if not X_list:
        return np.empty((0, eeg.shape[1], window), dtype=np.float32), \
               np.empty((0, joints.shape[1]), dtype=np.float32)
    return np.stack(X_list).astype(np.float32), \
           np.stack(y_list).astype(np.float32)


# Electrode positions

def _build_standard_positions() -> np.ndarray:
    """
    Returns approximate 3-D positions (mm) for 64 EEG channels in a standard
    BrainProducts layout projected onto a unit sphere of radius 85 mm.
    Only the 59 kept channels (after removing EOG_CHAN_INDICES) are returned.
    Shape: [59, 3]
    """
    az_el_64 = [
        (0,   0),   # Cz
        (180, 18),  # Fz
        (0,  18),   # Pz
        (270, 18),  # C3
        (90,  18),  # C4
        (225, 18),  # F3
        (135, 18),  # F4
        (315, 18),  # P3
        (45,  18),  # P4
        (180, 36),  # Fpz
        (0,   36),  # Oz
        (270, 36),  # T7
        (90,  36),  # T8
        (225, 36),  # F7
        (135, 36),  # F8
        (315, 36),  # P7
        (45,  36),  # P8
        (247, 28),  # FC5
        (113, 28),  # FC6
        (203, 28),  # FC1
        (157, 28),  # FC2
        (293, 28),  # CP5
        (67,  28),  # CP6
        (247, 28),  # FT9
        (113, 28),  # FT10
        (180, 52),  # AF7
        (0,   52),  # O1
        (270, 52),  # TP7
        (90,  52),  # TP8
        (225, 52),  # F5
        (135, 52),  # F6
        (315, 52),  # P5
        (45,  52),  # P6
        *[(i * (360/31), 72) for i in range(31)],
    ]
    r = 85.0
    positions = np.zeros((64, 3), dtype=np.float32)
    for i, (az, el) in enumerate(az_el_64):
        az_r = np.radians(az)
        el_r = np.radians(el)
        positions[i, 0] = r * np.sin(el_r) * np.cos(az_r)
        positions[i, 1] = r * np.sin(el_r) * np.sin(az_r)
        positions[i, 2] = r * np.cos(el_r)
    keep = [i for i in range(64) if i not in EOG_CHAN_INDICES]
    return positions[keep]


def build_adjacency_matrix(positions: Optional[np.ndarray] = None,
                            radius: float = RADIUS_MM) -> np.ndarray:
    """Build a binary adjacency matrix A. Self-loops added via eq.1 in train.py."""
    if positions is None:
        positions = _build_standard_positions()
    dist_matrix = cdist(positions, positions, metric='euclidean')
    A = (dist_matrix <= radius).astype(np.float32)
    np.fill_diagonal(A, 0.0)
    return A


# PyTorch Dataset

class MoBISessionDataset(Dataset):
    """Dataset for a single (subject, session, split) tuple."""

    def __init__(self, subject: str, session: str, split: str,
                 data_dir: Path = DATA_DIR, verbose: bool = True):
        assert split in ("train", "val", "test")
        self.subject = subject
        self.session = session
        self.split   = split

        folder = data_dir / f"{subject}-{session}"
        if not folder.exists():
            raise FileNotFoundError(f"Session folder not found: {folder}")

        if verbose:
            print(f"  Loading {subject}-{session} [{split}] ...", flush=True)

        ts_eeg,    eeg_raw    = _read_eeg   (folder / "eeg.txt")
        ts_joints, joints_raw = _read_joints(folder / "joints.txt")

        dt = np.median(np.diff(ts_eeg[:500]))
        fs = 1.0 / dt

        eeg_pp, joints_pp = _preprocess_session(eeg_raw, joints_raw, fs)

        total_samps = len(eeg_pp)
        train_samps = int(TRAIN_MIN * 60 * TARGET_FS)
        val_samps   = int(VAL_MIN   * 60 * TARGET_FS)
        test_end    = min(total_samps,
                          train_samps + val_samps + int(TEST_MIN * 60 * TARGET_FS))

        if split == "train":
            eeg_split    = eeg_pp   [:train_samps]
            joints_split = joints_pp[:train_samps]
        elif split == "val":
            s = train_samps
            e = train_samps + val_samps
            eeg_split    = eeg_pp   [s:e]
            joints_split = joints_pp[s:e]
        else:
            s = train_samps + val_samps
            eeg_split    = eeg_pp   [s:test_end]
            joints_split = joints_pp[s:test_end]

        self.X, self.y = _extract_windows(eeg_split, joints_split)

        if verbose:
            print(f"    -> {len(self.X)} windows, "
                  f"EEG: {eeg_split.shape}, joints: {joints_split.shape}", flush=True)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.y[idx])


class MoBIDataset(Dataset):
    """Aggregates data across all (subject, session) pairs for a given split."""

    def __init__(self, subjects: List[str] = SUBJECTS,
                 sessions: List[str] = SESSIONS, split: str = "train",
                 data_dir: Path = DATA_DIR, verbose: bool = True):
        self.datasets: List[MoBISessionDataset] = []
        for subj in subjects:
            for sess in sessions:
                folder = data_dir / f"{subj}-{sess}"
                if not folder.exists():
                    if verbose:
                        print(f"  Skipping missing: {subj}-{sess}")
                    continue
                try:
                    ds = MoBISessionDataset(subj, sess, split, data_dir, verbose)
                    if len(ds) > 0:
                        self.datasets.append(ds)
                except Exception as exc:
                    print(f"  Error loading {subj}-{sess}: {exc}")

        if self.datasets:
            self.X = np.concatenate([d.X for d in self.datasets], axis=0)
            self.y = np.concatenate([d.y for d in self.datasets], axis=0)
        else:
            self.X = np.empty((0, N_CHANNELS, WINDOW_SAMPS), dtype=np.float32)
            self.y = np.empty((0, N_JOINTS),                 dtype=np.float32)

        if verbose:
            print(f"\n[MoBIDataset-{split}] Total windows: {len(self.X)}")

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        x = torch.from_numpy(self.X[idx]).unsqueeze(0)  # [C, T] -> [1, C, T]
        y = torch.from_numpy(self.y[idx])
        return x, y


# DataLoader factory

def get_dataloaders(subjects: List[str] = SUBJECTS,
                    sessions: List[str] = SESSIONS,
                    batch_size: int      = 100,
                    num_workers: int     = 0,
                    data_dir: Path       = DATA_DIR,
                    verbose: bool        = True
                    ) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Build train / val / test DataLoaders."""
    print("Building Train dataset ...")
    train_ds = MoBIDataset(subjects, sessions, "train", data_dir, verbose)
    print("Building Val   dataset ...")
    val_ds   = MoBIDataset(subjects, sessions, "val",   data_dir, verbose)
    print("Building Test  dataset ...")
    test_ds  = MoBIDataset(subjects, sessions, "test",  data_dir, verbose)

    _pin = torch.cuda.is_available()
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=_pin, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=_pin)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=_pin)

    return train_dl, val_dl, test_dl


## MODEL — `model.py`

In [ ]:
%%writefile /kaggle/working/model.py
"""
model.py
--------
EEG2GAIT: A Hierarchical Graph Convolutional Network for EEG-Based Gait Decoding.

Paper architecture (Section III, Figure 2):
  1. LTL  - Local Temporal Learner (temporal Conv2d)
  2. GCM  - Graph Construction Module (learnable adjacency)
  3. HGP  - Hierarchical GCN Pyramid (K=2 branches)
  4. GSL  - Global Spatial Learner (depth-wise conv)
  5. FFN  - Feature Fusion Network (3 conv blocks [50,100,200])
  6. GTL  - Global Temporal Learner (multi-head self-attention)
  7. OUT  - Task-specific output layer
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from .config import (
        N_CHANNELS, WINDOW_SAMPS, N_JOINTS,
        F_FILTERS, LTL_KERNEL, HGP_DEPTHS,
        DROPOUT_P, POOL_WIDTH, KERNEL_WIDTH,
        FFN_FILTERS, GTL_HEADS, GTL_DROPOUT,
    )
except ImportError:
    from config import (
        N_CHANNELS, WINDOW_SAMPS, N_JOINTS,
        F_FILTERS, LTL_KERNEL, HGP_DEPTHS,
        DROPOUT_P, POOL_WIDTH, KERNEL_WIDTH,
        FFN_FILTERS, GTL_HEADS, GTL_DROPOUT,
    )


class Conv2dWithConstraint(nn.Conv2d):
    """Conv2d with per-filter L2 max-norm weight renormalization."""
    def __init__(self, *args, max_norm=2, **kwargs):
        self.max_norm = max_norm
        super().__init__(*args, **kwargs)

    def forward(self, x):
        self.weight.data = torch.renorm(
            self.weight.data, p=2, dim=0, maxnorm=self.max_norm
        )
        return super().forward(x)


# 1. LTL
class LocalTemporalLearner(nn.Module):
    """Input: [B, 1, C, T]  Output: [B, F, C, T]"""
    def __init__(self, n_filters=F_FILTERS, kernel=LTL_KERNEL):
        super().__init__()
        self.net = nn.Sequential(
            nn.ZeroPad2d(((kernel - 1) // 2, kernel // 2, 0, 0)),
            Conv2dWithConstraint(1, n_filters, (1, kernel), max_norm=2),
            nn.BatchNorm2d(n_filters),
            nn.ELU(),
        )

    def forward(self, x):
        return self.net(x)


# 2. GCM
class GraphConstructionModule(nn.Module):
    """Learnable adjacency matrix (paper Sec. III-C)."""
    def __init__(self, n_channels=N_CHANNELS, A_init=None):
        super().__init__()
        if A_init is not None:
            self.A = nn.Parameter(A_init.float())
        else:
            self.A = nn.Parameter(torch.eye(n_channels))

    def forward(self):
        A = torch.relu(self.A)
        D_raw = A.sum(dim=1)
        mask = (D_raw == 0).float()
        D = D_raw + mask
        D_inv_sqrt = D.pow(-0.5)
        return D_inv_sqrt.unsqueeze(1) * A * D_inv_sqrt.unsqueeze(0)


# 3. HGP
class GraphConvLayer(nn.Module):
    """Single GCN layer: H' = ReLU(A_norm . H . W + b)"""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, A_norm):
        x = x.permute(0, 3, 2, 1)          # [B, T, C, F]
        x = torch.matmul(A_norm, x)         # graph diffusion
        x = self.linear(x)
        x = F.relu(x)
        return x.permute(0, 3, 2, 1)        # [B, F_out, C, T]


class GCNBranch(nn.Module):
    """One HGP branch with its own learnable A."""
    def __init__(self, in_f, out_f, depth, n_channels=N_CHANNELS, A_init=None):
        super().__init__()
        self.gcm = GraphConstructionModule(n_channels, A_init)
        layers = []
        for i in range(depth):
            layers.append(GraphConvLayer(in_f if i == 0 else out_f, out_f))
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        A = self.gcm()
        for layer in self.layers:
            x = layer(x, A)
        return x


class HierarchicalGCNPyramid(nn.Module):
    """K parallel GCN branches at increasing depths + residual."""
    def __init__(self, in_f=F_FILTERS, out_f=F_FILTERS, depths=None,
                 n_channels=N_CHANNELS, A_init=None):
        super().__init__()
        if depths is None:
            depths = HGP_DEPTHS
        self.branches = nn.ModuleList([
            GCNBranch(in_f, out_f, d, n_channels, A_init) for d in depths
        ])

    def forward(self, x):
        branch_outs = [b(x) for b in self.branches]
        return torch.cat(branch_outs + [x], dim=1)


# 4. GSL
class GlobalSpatialLearner(nn.Module):
    """Depth-wise conv spanning all C channels. Input: [B,F,C,T] Output: [B,F,1,T//3]"""
    def __init__(self, n_filters, n_channels=N_CHANNELS, dropout=DROPOUT_P):
        super().__init__()
        self.net = nn.Sequential(
            Conv2dWithConstraint(n_filters, n_filters, (n_channels, 1),
                                 bias=False, max_norm=2),
            nn.BatchNorm2d(n_filters),
            nn.ELU(),
            nn.Dropout(p=dropout),
            nn.AvgPool2d((1, 3), stride=(1, 3)),
        )

    def forward(self, x):
        return self.net(x)


# 5. FFN
class FeatureFusionNetwork(nn.Module):
    """3 conv blocks for feature refinement + temporal downsampling."""
    def __init__(self, in_f, filter_list=None, kernel=KERNEL_WIDTH,
                 pool=POOL_WIDTH, dropout=DROPOUT_P):
        super().__init__()
        if filter_list is None:
            filter_list = FFN_FILTERS
        blocks = []
        prev = in_f
        for f in filter_list:
            blocks.append(nn.Sequential(
                nn.Dropout(p=dropout),
                nn.ZeroPad2d(((kernel - 1) // 2, kernel // 2, 0, 0)),
                Conv2dWithConstraint(prev, f, (1, kernel), bias=False, max_norm=2),
                nn.BatchNorm2d(f),
                nn.ELU(),
                nn.MaxPool2d((1, pool), stride=(1, pool)),
            ))
            prev = f
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)


# 6. GTL
class GlobalTemporalLearner(nn.Module):
    """Multi-head self-attention over temporal dimension with residual."""
    def __init__(self, embed_dim, n_heads=GTL_HEADS, dropout=GTL_DROPOUT):
        super().__init__()
        self.attn    = nn.MultiheadAttention(embed_dim, n_heads,
                                             dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, Fdim, _, T = x.shape
        seq = x.squeeze(2).permute(0, 2, 1)             # [B, T, F]
        attn_out, _ = self.attn(seq, seq, seq)
        out = self.norm(seq + self.dropout(attn_out))
        return out.permute(0, 2, 1).unsqueeze(2)


# Main Model
class EEG2Gait(nn.Module):
    """
    Full EEG2GAIT model.
    Input:  [B, 1, C, T]
    Output: [B, n_joints]
    """
    def __init__(self, n_channels: int = N_CHANNELS, n_time: int = WINDOW_SAMPS,
                 n_joints: int = N_JOINTS, A_init: torch.Tensor = None):
        super().__init__()
        n_hgp_branches = len(HGP_DEPTHS)
        n_gsl_in = F_FILTERS * (n_hgp_branches + 1)

        self.ltl = LocalTemporalLearner(F_FILTERS, LTL_KERNEL)
        self.hgp = HierarchicalGCNPyramid(F_FILTERS, F_FILTERS, HGP_DEPTHS,
                                           n_channels, A_init)
        self.gsl = GlobalSpatialLearner(n_gsl_in, n_channels)
        self.ffn = FeatureFusionNetwork(n_gsl_in, FFN_FILTERS)
        self.gtl = GlobalTemporalLearner(FFN_FILTERS[-1], GTL_HEADS, GTL_DROPOUT)

        T_final = n_time
        for _ in range(1 + len(FFN_FILTERS)):
            T_final = T_final // POOL_WIDTH
        self.output = Conv2dWithConstraint(
            FFN_FILTERS[-1], n_joints, (1, T_final * 2), max_norm=0.5
        )

    def forward(self, x):
        x = self.ltl(x)
        x = self.hgp(x)
        x = self.gsl(x)
        x = self.ffn(x)
        gtl_in = x
        x = self.gtl(x)
        x = torch.cat([gtl_in, x], dim=3)
        x = self.output(x)
        return x.squeeze(3).squeeze(2)


def build_model(A_init=None, **kwargs) -> EEG2Gait:
    """Construct EEG2Gait. Pass A_init for GCM initialization."""
    return EEG2Gait(A_init=A_init, **kwargs)


## LOSS — `loss.py`

In [ ]:
%%writefile /kaggle/working/loss.py
"""
loss.py
-------
Hybrid Temporal-Spectral Reward (HTSR) Loss (paper Sec. III-I).

L_total = alpha * L_freq_reward + (1 - alpha) * L_time_reward
Default: alpha=0.5, beta=0.1
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from .config import ALPHA, BETA, EPSILON
except ImportError:
    from config import ALPHA, BETA, EPSILON


class HTSRLoss(nn.Module):
    """Hybrid Temporal-Spectral Reward Loss."""

    def __init__(self, alpha: float = ALPHA, beta: float = BETA, eps: float = EPSILON):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.eps   = eps

    def _reward(self, loss_val: torch.Tensor) -> torch.Tensor:
        L     = loss_val.clamp(min=1e-12)
        inner = (1.0 - torch.exp(-L) + self.eps).clamp(min=self.eps)
        return L + self.beta * torch.log(inner)

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        # Time-domain
        L_time   = F.mse_loss(y_pred, y_true)
        L_time_r = self._reward(L_time)

        # Frequency-domain
        Y_hat_freq = torch.fft.rfft(y_pred, dim=1)
        Y_freq     = torch.fft.rfft(y_true, dim=1)
        L_freq     = F.l1_loss(Y_hat_freq.abs(), Y_freq.abs())
        L_freq_r   = self._reward(L_freq)

        return self.alpha * L_freq_r + (1.0 - self.alpha) * L_time_r


## METRICS — `metrics.py`

In [ ]:
%%writefile /kaggle/working/metrics.py
"""
metrics.py
----------
Evaluation metrics for EEG2GAIT gait angle prediction.
Pearson r, R-squared, MAE (per-joint and averaged).
"""

import torch
import numpy as np
from typing import Dict, Tuple

try:
    from .config import JOINT_NAMES, N_JOINTS
except ImportError:
    from config import JOINT_NAMES, N_JOINTS


def pearson_r(y_pred: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    """Pearson correlation coefficient per joint."""
    r_vals = []
    for j in range(y_pred.shape[1]):
        p = y_pred[:, j]
        t = y_true[:, j]
        if np.std(p) < 1e-8 or np.std(t) < 1e-8:
            r_vals.append(0.0)
        else:
            corr = np.corrcoef(p, t)[0, 1]
            r_vals.append(float(corr) if not np.isnan(corr) else 0.0)
    return np.array(r_vals)


def r2_score(y_pred: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    """R-squared per joint."""
    ss_res = ((y_true - y_pred) ** 2).sum(axis=0)
    ss_tot = ((y_true - y_true.mean(axis=0)) ** 2).sum(axis=0)
    r2 = np.where(ss_tot < 1e-12, 0.0, 1.0 - ss_res / ss_tot)
    return r2


def mae_score(y_pred: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    """Mean Absolute Error per joint."""
    return np.abs(y_pred - y_true).mean(axis=0)


def compute_metrics(y_pred: np.ndarray, y_true: np.ndarray) -> Dict[str, float]:
    """Compute all metrics and return as a flat dict."""
    r   = pearson_r(y_pred, y_true)
    r2  = r2_score (y_pred, y_true)
    mae = mae_score(y_pred, y_true)

    results: Dict[str, float] = {}
    for j, name in enumerate(JOINT_NAMES):
        results[f"r_{name}"]   = float(r[j])
        results[f"r2_{name}"]  = float(r2[j])
        results[f"mae_{name}"] = float(mae[j])

    results["r_mean"]   = float(r.mean())
    results["r2_mean"]  = float(r2.mean())
    results["mae_mean"] = float(mae.mean())
    return results


def evaluate_loader(model, loader, device: str) -> Dict[str, float]:
    """Run inference over the entire DataLoader and compute metrics."""
    import torch
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            y_hat = model(X)
            preds.append(y_hat.cpu().numpy())
            targets.append(y.numpy())

    y_pred = np.concatenate(preds,   axis=0)
    y_true = np.concatenate(targets, axis=0)
    return compute_metrics(y_pred, y_true)


## TRAIN — `train.py`

In [ ]:
%%writefile /kaggle/working/train.py
"""
train.py
--------
Training loop for EEG2GAIT.

Features:
  - Adam optimiser (lr=0.001)
  - HTSR custom loss
  - Early stopping: patience=30, monitored metric = mean Pearson r on val set
  - Per-epoch logging
  - Checkpoint saving (best val r model)
  - Final test-set evaluation with per-joint breakdown
"""

import os
import sys
import time
import json
import copy
import random
import argparse
from pathlib import Path

import torch
import numpy as np

# Kaggle/PyTorch compatibility fix
try:
    import torch._utils
except Exception:
    pass

# Path setup
SRC_DIR = Path(__file__).parent
sys.path.insert(0, str(SRC_DIR))

from config import (
    DATA_DIR, OUTPUT_DIR, SUBJECTS, SESSIONS,
    BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE,
    SEED, NUM_WORKERS, DEVICE,
    N_CHANNELS, WINDOW_SAMPS, N_JOINTS,
)
from dataset import get_dataloaders, build_adjacency_matrix, _build_standard_positions
from model  import build_model
from loss   import HTSRLoss
from metrics import evaluate_loader, compute_metrics


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def log(msg: str, log_file=None):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    if log_file:
        log_file.write(line + "\n")
        log_file.flush()


def train(subjects=None, sessions=None, device_str=None,
          batch_size=BATCH_SIZE, lr=LR, max_epochs=MAX_EPOCHS,
          patience=PATIENCE, output_dir=OUTPUT_DIR):
    """Full training pipeline."""
    set_seed(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if subjects is None: subjects = SUBJECTS
    if sessions is None: sessions = SESSIONS

    # Device
    if device_str is None:
        device_str = DEVICE
    if device_str == "auto":
        if torch.cuda.is_available():
            device_str = "cuda"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device_str = "mps"
        else:
            device_str = "cpu"
    device = torch.device(device_str)
    print(f"\n{'='*60}")
    print(f"  EEG2GAIT Training Run")
    print(f"  Device:   {device}")
    print(f"  Subjects: {subjects}")
    print(f"  Sessions: {sessions}")
    print(f"{'='*60}\n")

    log_path = output_dir / "train_log.txt"
    log_file = open(log_path, "w")

    # Data
    log("Loading data ...", log_file)
    train_dl, val_dl, test_dl = get_dataloaders(
        subjects=subjects, sessions=sessions,
        batch_size=batch_size, num_workers=NUM_WORKERS,
        verbose=True
    )
    log(f"Train batches: {len(train_dl)} | "
        f"Val batches: {len(val_dl)} | "
        f"Test batches: {len(test_dl)}", log_file)

    # Adjacency matrix (GCM initialization, paper eq.1)
    log("Building adjacency matrix ...", log_file)
    positions = _build_standard_positions()
    A_np      = build_adjacency_matrix(positions)
    A_tensor  = torch.from_numpy(A_np)
    A_init = torch.relu(A_tensor + A_tensor.T) + torch.eye(A_tensor.size(0))

    # Model
    log("Building model ...", log_file)
    model = build_model(A_init=A_init).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"Trainable parameters: {n_params:,}", log_file)

    # Optimiser & loss
    try:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    except AttributeError:
        import importlib
        _utils = importlib.import_module("torch._utils")
        torch._utils = _utils
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = HTSRLoss()

    # Training state
    best_val_r    = -1.0
    best_epoch    = 0
    epochs_no_imp = 0
    best_state    = None
    history       = {"train_loss": [], "val_r": [], "val_r2": [], "val_mae": []}

    # Epoch loop
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()

        model.train()
        total_loss = 0.0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(X)
            loss  = criterion(y_hat, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / max(len(train_dl), 1)

        # Validate
        val_metrics = evaluate_loader(model, val_dl, device_str)
        val_r       = val_metrics["r_mean"]
        val_r2      = val_metrics["r2_mean"]
        val_mae     = val_metrics["mae_mean"]

        elapsed = time.time() - t0
        msg = (f"Epoch {epoch:03d}/{max_epochs} | "
               f"Loss: {avg_train_loss:.4f} | "
               f"Val r: {val_r:.4f} | "
               f"Val R2: {val_r2:.4f} | "
               f"Val MAE: {val_mae:.4f} | "
               f"Time: {elapsed:.1f}s")
        log(msg, log_file)

        history["train_loss"].append(avg_train_loss)
        history["val_r"].append(val_r)
        history["val_r2"].append(val_r2)
        history["val_mae"].append(val_mae)

        # Early stopping
        if val_r > best_val_r:
            best_val_r    = val_r
            best_epoch    = epoch
            epochs_no_imp = 0
            best_state    = copy.deepcopy(model.state_dict())
            ckpt_path     = output_dir / "best_model.pt"
            torch.save({
                "epoch":      epoch,
                "state_dict": best_state,
                "val_r":      best_val_r,
                "val_r2":     val_r2,
                "val_mae":    val_mae,
            }, ckpt_path)
            log(f"  New best model saved (val r={best_val_r:.4f})", log_file)
        else:
            epochs_no_imp += 1

        if epochs_no_imp >= patience:
            log(f"\n  Early stopping triggered after {epoch} epochs "
                f"(best epoch={best_epoch}, best val r={best_val_r:.4f})", log_file)
            break

    # Load best model
    log(f"\nLoading best model (epoch {best_epoch}) ...", log_file)
    if best_state is not None:
        model.load_state_dict(best_state)

    # Test evaluation
    log("\nEvaluating on TEST set ...", log_file)
    test_metrics = evaluate_loader(model, test_dl, device_str)

    log("\n" + "="*60, log_file)
    log("TEST RESULTS", log_file)
    log("="*60, log_file)
    log(f"  Mean Pearson r : {test_metrics['r_mean']:.4f}", log_file)
    log(f"  Mean R2        : {test_metrics['r2_mean']:.4f}", log_file)
    log(f"  Mean MAE       : {test_metrics['mae_mean']:.4f}", log_file)
    log("-"*60, log_file)
    log("  Per-joint breakdown:", log_file)
    from config import JOINT_NAMES
    for name in JOINT_NAMES:
        log(f"    {name:5s}  r={test_metrics[f'r_{name}']:.4f}  "
            f"R2={test_metrics[f'r2_{name}']:.4f}  "
            f"MAE={test_metrics[f'mae_{name}']:.4f}", log_file)
    log("="*60, log_file)

    # Save results
    results = {
        "best_epoch":    best_epoch,
        "best_val_r":    best_val_r,
        "test_metrics":  test_metrics,
        "history":       history,
    }
    results_path = output_dir / "results.json"
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    log(f"\nResults saved to: {results_path}", log_file)

    log_file.close()
    return model, results


def parse_args():
    p = argparse.ArgumentParser(description="Train EEG2GAIT on the MoBI dataset")
    p.add_argument("--subjects",    nargs="+", default=None)
    p.add_argument("--sessions",    nargs="+", default=None)
    p.add_argument("--device",      default="auto")
    p.add_argument("--batch-size",  type=int, default=BATCH_SIZE)
    p.add_argument("--lr",          type=float, default=LR)
    p.add_argument("--max-epochs",  type=int, default=MAX_EPOCHS)
    p.add_argument("--patience",    type=int, default=PATIENCE)
    p.add_argument("--output-dir",  default=str(OUTPUT_DIR))
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    train(
        subjects   = args.subjects,
        sessions   = args.sessions,
        device_str = args.device,
        batch_size = args.batch_size,
        lr         = args.lr,
        max_epochs = args.max_epochs,
        patience   = args.patience,
        output_dir = args.output_dir,
    )


## Kaggle Path Patch

Override `DATA_DIR` and `OUTPUT_DIR` in the imported config module.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')

import config as cfg
from pathlib import Path

cfg.DATA_DIR   = Path('/kaggle/input/datasets/jwangldan09/eeg2gait-fall-prediction-dataset/RepositoryData')
cfg.OUTPUT_DIR = Path('/kaggle/working/outputs')
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))

## Run Training

Trains all 8 subjects × 3 sessions.  
Best checkpoint → `/kaggle/working/outputs/best_model.pt`

In [ ]:
import torch
from train import train

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device}')

model, results = train(
    device_str  = device,
    batch_size  = 100,
    max_epochs  = 50,
    patience    = 30,
    output_dir  = '/kaggle/working/outputs',
)

## Results

In [ ]:
import json

with open('/kaggle/working/outputs/results.json') as f:
    res = json.load(f)

tm = res['test_metrics']
print('='*50)
print('TEST SET RESULTS')
print('='*50)
print(f"Mean Pearson r : {tm['r_mean']:.4f}")
print(f"Mean R2        : {tm['r2_mean']:.4f}")
print(f"Mean MAE       : {tm['mae_mean']:.4f}")
print('-'*50)
joints = ['GHR','GKR','GAR','GHL','GKL','GAL']
print(f"{'Joint':6s}  {'r':>7s}  {'R2':>7s}  {'MAE':>7s}")
print('-'*34)
for j in joints:
    print(f"{j:6s}  {tm[f'r_{j}']:7.4f}  {tm[f'r2_{j}']:7.4f}  {tm[f'mae_{j}']:7.4f}")
print('='*50)
print(f"Best epoch: {res['best_epoch']} | Best val r: {res['best_val_r']:.4f}")